# M2_E5 — TF-IDF name/address blocking + LightGBM pair matching

Indexes the combined Source 2 + Source 3 universe with two sparse TF-IDF matrices
(one over normalized `business_name`, one over normalized `business_address`), retrieves
the Top-K candidates per Source 1 row from each channel, unions them into one candidate
set, and trains a LightGBM matcher on the resulting pairs.

**The dataset is read from S3 and staged onto the instance's local disk first.** The store
builder seeks through the TSVs and the ground-truth index memory-maps byte offsets computed
from them, so both need real seekable files rather than a streaming object handle.

---

## Runtime and instance sizing

Retrieval is the dominant cost and it is embarrassingly parallel over query batches: each
worker memory-maps the target store read-only and scores its own batches, so `--workers`
scales close to linearly with vCPU count until memory bandwidth saturates.

| Stage | What dominates | Notes |
|---|---|---|
| Stage 1 stage S3 | network + disk | one-time, reused by every later cell |
| Stage 2 smoke | seconds | end-to-end proof, tiny derived dataset |
| Stage 3 K sweep | store build + retrieval + LightGBM | the expensive one |
| Stage 4 final inference | store build + retrieval | builds a second store for the test split |

Every stage runs as a **subprocess**, which matters: it guarantees the multi-gigabyte
memory-mapped store and the LightGBM models are released when the stage ends, instead of
accumulating across cells in the notebook kernel.

**Peak memory is dominated by the in-flight similarity blocks:**
`workers x query_batch_rows x target_block_rows x 4 bytes`. The defaults
(`--query-batch-rows 1024 --target-block-rows 8192`) cost ~32 MB per worker, so even
32 workers stay well under 2 GB there. Add the store itself (~1.3 GB of CSR for the full
10.3M-row S2+S3 universe) and the pandas/Arrow overhead of a gathered frame.

If you hit memory pressure, lower `--workers` first; that knob scales peak memory linearly
and runtime inversely.

## Stage 0 — configuration

Everything downstream reads these two cells.

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

# ---------------------------------------------------------------- project
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "experiments" / "M2_E5").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise SystemExit("run this notebook from the ML-Challenge project root")
    PROJECT_ROOT = PROJECT_ROOT.parent
print("project root:", PROJECT_ROOT)

# ---------------------------------------------------------------- S3 dataset
# Point S3_PREFIX at wherever you uploaded the dataset. Either layout works:
#   s3://bucket/ml-challenge/            -> <prefix>/student_resource/dataset/{train,test}
#   s3://bucket/ml-challenge/dataset     -> <prefix>/{train,test}
S3_BUCKET = "my-bucket"
S3_PREFIX = "ml-challenge"
USE_S3 = True  # set False to read the dataset already on local disk

LOCAL_DATA_ROOT = PROJECT_ROOT
if USE_S3:
    TRAIN_DIR = f"s3://{S3_BUCKET}/{S3_PREFIX}/student_resource/dataset/train"
    TEST_DIR = f"s3://{S3_BUCKET}/{S3_PREFIX}/student_resource/dataset/test"
else:
    TRAIN_DIR = str(PROJECT_ROOT / "student_resource" / "dataset" / "train")
    TEST_DIR = str(PROJECT_ROOT / "student_resource" / "dataset" / "test")

# ---------------------------------------------------------------- resources
# 0 means "every core but one". Retrieval scales with this, and so does peak memory.
WORKERS = max(1, (os.cpu_count() or 2) - 1)
QUERY_BATCH_ROWS = 1_024      # per worker task; raise for fewer, larger tasks
TARGET_BLOCK_ROWS = 8_192     # target rows per similarity block
K_VALUES = "5,10,20,30,50"   # swept against validation macro F0.5
SEED = 42

# Local scratch. The S3 stage and the temporary stores both need real disk space:
# roughly 10 GB for the staged TSVs plus ~4 GB for a full S2+S3 TF-IDF store.
SCRATCH = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.home() / "m2e5"
STAGE_DIR = SCRATCH / "s3_stage"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "M2_E5" / "artifacts"
SUBMISSION_DIR = SCRATCH / "submission"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"cores={os.cpu_count()} workers={WORKERS}")
print("train:", TRAIN_DIR)
print("test :", TEST_DIR)


def run_cli(*args, label: str | None = None):
    """Run the experiment CLI in a subprocess so its memory is fully released."""
    cmd = [sys.executable, "-m", "experiments.M2_E5.run_experiment"] + [str(a) for a in args]
    print("\n" + "=" * 78, flush=True)
    print(f"[{(label or 'stage')}] {' '.join(cmd)}", flush=True)
    print("=" * 78, flush=True)
    started = time.perf_counter()
    completed = subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)
    elapsed = time.perf_counter() - started
    if completed.returncode != 0:
        raise SystemExit(f"{label or 'stage'} failed with exit code {completed.returncode}")
    print(f"\n[{(label or 'stage')}] finished in {elapsed / 60:.1f} min", flush=True)
    return elapsed


COMMON = [
    "--train-dir", TRAIN_DIR,
    "--test-dir", TEST_DIR,
    "--output-dir", OUTPUT_DIR,
    "--stage-dir", STAGE_DIR,
    "--workers", WORKERS,
    "--query-batch-rows", QUERY_BATCH_ROWS,
    "--target-block-rows", TARGET_BLOCK_ROWS,
    "--seed", SEED,
]
print("common args:", COMMON)

## Stage 1 — stage the dataset from S3

Downloads the seven TSVs onto the instance's local disk. Downloads land on a `.part` file
and are renamed only once complete, and already-staged files are reused, so this cell is
safe to re-run after an interruption and cheap on a second pass.

In [ ]:
import boto3  # noqa: F401  (fails fast here rather than mid-run)

client = boto3.client("s3")
token = client.get_paginator("list_objects_v2")
prefix = S3_PREFIX.strip("/") + "/"
print(f"objects under s3://{S3_BUCKET}/{prefix}")
for page in token.paginate(Bucket=S3_BUCKET, Prefix=prefix):
    for item in page.get("Contents", []):
        size_gb = item["Size"] / 1e9
        if item["Key"].endswith(".tsv"):
            print(f"  {item['Key']:<70} {size_gb:>6.2f} GB")

## Stage 2 — smoke test

Derives a small dataset from the real files that **keeps genuine matches** (every Source 1
row and its true candidates are preserved) and pads it with filler rows, so the smoke test
exercises recall instead of measuring recall against an empty target set. Writes to a
temporary directory and deletes it. Run this before committing to the expensive stages.

In [ ]:
run_cli(*COMMON, "--smoke-test", label="smoke")

## Stage 3 — K sweep on the training split

Retrieves once at K=50 and derives every smaller K from the per-channel ranks, so the sweep
costs one retrieval pass rather than five. One LightGBM model is fit per K and scored against
a deterministic match-count-stratified holdout; the winner is the best validation macro F0.5
and is promoted to `selected_model.txt`.

In [ ]:
run_cli(*COMMON, "--k-values", K_VALUES, label="k-sweep")

import json

metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text(encoding="utf-8"))
print("selected K:", metrics["selected_k"])
print("\nper-K validation (selected K in brackets):")
for k, s in metrics["per_k"].items():
    mark = "  <- selected" if int(k) == metrics["selected_k"] else ""
    print(
        f"K={k:>3}  blocking_recall={s['blocking_recall']:.4f}  "
        f"macro_f0_5={s['macro_f0_5']:.4f}  "
        f"P_micro={s['matching_precision_micro']:.4f}  "
        f"R_micro={s['matching_recall_micro']:.4f}  "
        f"cand/row={s['candidate_count_average']:.1f}"
        f"{mark}"
    )
print(
    "\nblocking_recall is the ceiling: the ranker can only score pairs that "
    "retrieval surfaced."
)

## Stage 4 — final test inference

**This cell does nothing unless you flip the toggle.** Final inference is the single most
expensive stage: it builds a second TF-IDF store over the test S2+S3 universe and scores
every test Source 1 row.

It writes `candidate_pairs.tsv` and `matching_results.tsv` with the headers
`source1_entity_id\tcandidate_entity_ids` and `source1_entity_id\tmatched_entity_ids`.
Every test Source 1 ID appears exactly once, and only retrieved candidates can be predicted.

In [ ]:
RUN_FINAL_INFERENCE = False  # flip to True when you actually want the submission

if RUN_FINAL_INFERENCE:
    run_cli(
        *COMMON,
        "--final-inference",
        "--submission-dir", SUBMISSION_DIR,
        label="final-inference",
    )
else:
    print("skipped: set RUN_FINAL_INFERENCE = True to generate the submission")

In [ ]:
candidates = SUBMISSION_DIR / "candidate_pairs.tsv"
results = SUBMISSION_DIR / "matching_results.tsv"

if results.is_file():
    with results.open(encoding="utf-8") as handle:
        header = handle.readline().rstrip("\n")
        sample = [handle.readline().rstrip("\n") for _ in range(5)]
    print("matching_results.tsv header:", header)
    for line in sample:
        print("  ", line[:160])
    print("\nrows:", sum(1 for _ in results.open(encoding='utf-8')) - 1)
else:
    print("no submission yet:", results)

## Stage 5 — upload results back to S3

The submission TSVs are the deliverable. Metrics and the selected model are small enough to
keep alongside them.

In [ ]:
UPLOAD_TO_S3 = False  # flip to True once the submission looks right

if UPLOAD_TO_S3 and results.is_file():
    out_prefix = f"{S3_PREFIX.strip('/')}/m2e5/output"
    for path in (candidates, results):
        key = f"{out_prefix}/{path.name}"
        print(f"uploading {path} -> s3://{S3_BUCKET}/{key}", flush=True)
        client.upload_file(str(path), S3_BUCKET, key)
    for name in ("metrics.json", "config.json"):
        path = OUTPUT_DIR / name
        if path.is_file():
            client.upload_file(str(path), S3_BUCKET, f"{out_prefix}/{name}")
            print(f"uploading {name}")
    print("done")
else:
    print("skipped: set UPLOAD_TO_S3 = True after validating the submission")